In [1]:
p = 0.2

In [2]:
import numpy as np
from qiskit.circuit import QuantumCircuit


class BernoulliA(QuantumCircuit):
    """A circuit representing the Bernoulli A operator."""

    def __init__(self, probability):
        super().__init__(1)  # circuit on 1 qubit

        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)


class BernoulliQ(QuantumCircuit):
    """A circuit representing the Bernoulli Q operator."""

    def __init__(self, probability):
        super().__init__(1)  # circuit on 1 qubit

        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        # implement the efficient power of Q
        q_k = QuantumCircuit(1)
        q_k.ry(2 * k * self._theta_p, 0)
        return q_k

In [3]:
A = BernoulliA(p)
Q = BernoulliQ(p)

In [4]:
from qiskit.algorithms import EstimationProblem

problem = EstimationProblem(
    state_preparation=A,  # A operator
    grover_operator=Q,  # Q operator
    objective_qubits=[0],  # the "good" state Psi1 is identified as measuring |1> in qubit 0
)

<ipython-input-4-254369b6a41a>:1: DeprecationWarning: ``qiskit.algorithms`` has been migrated to an independent package: https://github.com/qiskit-community/qiskit-algorithms. The ``qiskit.algorithms`` import path is deprecated as of qiskit-terra 0.25.0 and will be removed no earlier than 3 months after the release date. Please run ``pip install qiskit_algorithms`` and use ``import qiskit_algorithms`` instead.
  from qiskit.algorithms import EstimationProblem


In [10]:
from qiskit.primitives import Sampler

sampler = Sampler(options={'shots':2})

In [11]:
from qiskit.algorithms import IterativeAmplitudeEstimation

iae = IterativeAmplitudeEstimation(
    epsilon_target=0.01,  # target accuracy
    alpha=0.0001,  # width of the confidence interval
    sampler=sampler,
)
iae_result = iae.estimate(problem)

print("Estimate:", iae_result.estimation)

Estimate: 0.2008964691168238


In [12]:
print(iae_result)

{   'alpha': 0.0001,
    'circuit_results': None,
    'confidence_interval': (0.1929380966304688, 0.2088548416031788),
    'confidence_interval_processed': (0.1929380966304688, 0.2088548416031788),
    'epsilon_estimated': 0.007958372486355003,
    'epsilon_estimated_processed': 0.007958372486355003,
    'epsilon_target': None,
    'estimate_intervals': [   [0.0, 1.0],
                              [0.0, 0.9973273875808728],
                              [0.0, 0.9483026846042557],
                              [0.0, 0.8612247577715847],
                              [0.0, 0.772629563496605],
                              [0.0, 0.6942354049029115],
                              [5.952400439213082e-07, 0.7209200716335475],
                              [5.102057736294797e-07, 0.6622845444139799],
                              [4.4643006619216047e-07, 0.6111539154635017],
                              [3.9682673530494494e-07, 0.5665915909525924],
                              [3.571440688